In [1]:
#loading important libraries
import kagglehub
import shutil
import json
import pickle
import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score,train_test_split,StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score,classification_report,roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


from dotenv import load_dotenv



seed=42
print('libraries imported successfully')

libraries imported successfully


In [2]:
kagglehub.login()

In [3]:
!kaggle competitions list

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [4]:
!kaggle competitions download -c titanic

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [5]:
#data loading
train=pd.read_csv('/teamspace/studios/this_studio/Titanic/data/train.csv')
test=pd.read_csv('/teamspace/studios/this_studio/Titanic/data/test.csv')


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [7]:
train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [8]:
train.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [9]:
train['Survived'].value_counts()

Survived
0    549
1    342
Name: count, dtype: int64

In [10]:
y=train['Survived']
X=train.drop('Survived',axis=1)


In [11]:
X['Cabin']=X['Cabin'].apply(lambda x:0 if pd.isna(x) else 1).astype('object')
test['Cabin']=test['Cabin'].apply(lambda x:0 if pd.isna(x) else 1).astype('object')


In [12]:
X.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)
test.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

In [13]:
numerical_cols=X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols=X.select_dtypes(include=[object]).columns.tolist()

In [14]:
categorical_cols

['Sex', 'Cabin', 'Embarked']

In [15]:
num_pipeline=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='median')),
    ('scale',StandardScaler())

])
cat_pipeline=Pipeline(steps=[
    ('impute',SimpleImputer(strategy='most_frequent')),
    ('encode',OrdinalEncoder())

])

preprocessor=ColumnTransformer(transformers=[
('num',num_pipeline,numerical_cols),
('cat',cat_pipeline,categorical_cols)
])


In [16]:
lgb_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(n_estimators=300, num_leaves=5, learning_rate=0.02, random_state=seed,force_col_wise=True))
])

In [17]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
scores = []
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    x_train, x_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train model
    lgb_model.fit(x_train, y_train)
    
    # Store out-of-fold predictions
    oof_preds[val_idx] = lgb_model.predict_proba(x_val)[:, 1]  # ← Use predict_proba for ROC-AUC!
    
    # Accumulate test predictions
    test_preds += lgb_model.predict_proba(test)[:, 1] / skf.n_splits
    
    # Calculate score for THIS fold only
    fold_score = roc_auc_score(y_val, oof_preds[val_idx])  # ← Score only on validation set!
    scores.append(fold_score)
    
    print(f"Fold {fold + 1}: {fold_score:.4f}")

# Overall OOF score (calculated AFTER the loop)
overall_oof_score = roc_auc_score(y, oof_preds)

print(f"\nFold scores: {scores}")
print(f"Mean CV: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
print(f"Overall OOF: {overall_oof_score:.4f}")

[LightGBM] [Info] Number of positive: 273, number of negative: 439
[LightGBM] [Info] Total Bins 202
[LightGBM] [Info] Number of data points in the train set: 712, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383427 -> initscore=-0.475028
[LightGBM] [Info] Start training from score -0.475028
Fold 1: 0.9122
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Total Bins 207
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.384292 -> initscore=-0.471371
[LightGBM] [Info] Start training from score -0.471371
Fold 2: 0.8840
[LightGBM] [Info] Number of positive: 274, number of negative: 439
[LightGBM] [Info] Total Bins 206
[LightGBM] [Info] Number of data points in the train set: 713, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.384292 -> initscore=-0.471371
[LightGBM] [Info] Start training from score -0.47137

In [23]:
rf_pipeline=Pipeline([
    ('preprocess',preprocessor),
    ('model',RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    ))
])

In [24]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
scores = []
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    x_train, x_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train model
    rf_pipeline.fit(x_train, y_train)
    
    # Store out-of-fold predictions
    oof_preds[val_idx] = rf_pipeline.predict_proba(x_val)[:, 1]  # ← Use predict_proba for ROC-AUC!
    
    # Accumulate test predictions
    test_preds += rf_pipeline.predict_proba(test)[:, 1] / skf.n_splits
    
    # Calculate score for THIS fold only
    fold_score = roc_auc_score(y_val, oof_preds[val_idx])  # ← Score only on validation set!
    scores.append(fold_score)
    
    print(f"Fold {fold + 1}: {fold_score:.4f}")

# Overall OOF score (calculated AFTER the loop)
overall_oof_score = roc_auc_score(y, oof_preds)

print(f"\nFold scores: {scores}")
print(f"Mean CV: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")
print(f"Overall OOF: {overall_oof_score:.4f}")

Fold 1: 0.9062
Fold 2: 0.8806
Fold 3: 0.8426
Fold 4: 0.8495
Fold 5: 0.8896

Fold scores: [0.9061923583662714, 0.8806149732620321, 0.8425802139037433, 0.8495320855614973, 0.8895758542746975]
Mean CV: 0.8737 (+/- 0.0241)
Overall OOF: 0.8708


In [2]:
import mlflow
client = mlflow.MlflowClient()
for rm in client.search_registered_models():
    print(rm.name)

sqlite:////teamspace/studios/this_studio/Titanic/Notebooks/mlflow.db
